# Ultralytics YOLO 학습 노트북

VSCode에서 셀 단위로 실행하며 Ultralytics를 익히는 노트북입니다.

**사용법**
- 셀 선택 후 `Shift + Enter` = 실행하고 다음 셀로
- `Ctrl + Enter` = 실행하고 그 자리에 머무름
- 우측 상단 커널이 **Python (ultralytics-practice)** 인지 꼭 확인하세요.

> 이 맥(Apple M2 Pro)은 CUDA가 없으므로 GPU 가속은 `device="mps"`를 씁니다.

## 0. 환경 확인
먼저 torch / ultralytics 버전과 어떤 device를 쓸 수 있는지 확인합니다.

In [ ]:
import os
# MPS에서 아직 미구현된 연산은 CPU로 자동 폴백 (import torch 전에 설정)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch, ultralytics

print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("CUDA 사용가능:", torch.cuda.is_available())
print("MPS 사용가능 :", torch.backends.mps.is_available())

# 앞으로 이 변수 하나만 바꾸면 전체 device가 바뀝니다
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("=> 사용할 DEVICE:", DEVICE)

## 1. 모델 불러오기

YOLO 모델은 두 가지 방식으로 만듭니다.

| 방식 | 코드 | 언제 |
|---|---|---|
| **사전학습 가중치** (`.pt`) | `YOLO("yolo26n.pt")` | 대부분의 경우 (전이학습·추론) |
| **구조만** (`.yaml`) | `YOLO("yolo26n.yaml")` | 밑바닥부터(scratch) 학습 실험 |

모델 크기: `n`(nano) < `s` < `m` < `l` < `x`. 뒤로 갈수록 정확↑ 속도↓.

In [ ]:
from ultralytics import YOLO

# 사전학습 모델 로드 (없으면 자동 다운로드)
model = YOLO("yolo11n.pt")

# 모델 구조 요약 보기
model.info()

## 2. 추론(Predict) — 가장 먼저 감 잡기

학습 없이 사전학습 모델로 바로 객체 탐지를 해봅니다. 입력은 URL / 파일경로 / 폴더 / numpy 배열 / 비디오 모두 가능.

In [ ]:
results = model.predict(
    "https://ultralytics.com/images/bus.jpg",
    device=DEVICE,
    conf=0.25,      # 신뢰도 임계값
    save=True,      # runs/detect/predict*/ 에 결과 이미지 저장
)
print("저장 위치:", results[0].save_dir)

### 2-1. Results 객체 뜯어보기

`predict`는 이미지마다 `Results` 객체를 담은 리스트를 돌려줍니다. 핵심은 `.boxes`.

In [ ]:
r = results[0]

boxes = r.boxes

print("탐지된 객체 수:", len(boxes))
print("클래스 이름 맵 :", r.names)   # {0: 'person', 5: 'bus', ...}
print()

for i in range(len(boxes)):
    cls_id = int(boxes.cls[i])
    conf   = float(boxes.conf[i])
    xyxy   = boxes.xyxy[i].tolist()   # [x1, y1, x2, y2] 픽셀 좌표
    print(f"{r.names[cls_id]:10} conf={conf:.2f}  box={[round(v,1) for v in xyxy]}")

In [ ]:
# 노트북 안에서 결과 이미지 바로 보기 (박스가 그려진 상태)
from PIL import Image
import io

im_bgr = r.plot(boxes=True, labels=False, conf=True, line_width=2)          # numpy 배열 (BGR)
 
Image.fromarray(im_bgr[..., ::-1])   # BGR -> RGB 로 뒤집어 표시

## 3. 학습(Train)

`coco8`은 이미지 8장짜리 **동작 확인용** 데이터셋입니다 (자동 다운로드).

자주 쓰는 인자:
- `data` : 데이터셋 yaml 경로
- `epochs` : 전체 데이터 반복 횟수
- `imgsz` : 입력 이미지 크기 (기본 640)
- `batch` : 배치 크기
- `device` : `"mps"` / `"cpu"` / (NVIDIA면 `0`)
- `patience` : 개선 없을 때 조기종료 대기 epoch

In [ ]:
model = YOLO("yolo26n.pt")

train_results = model.train(
    data="coco8.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=0,       # macOS에서는 0이 안전
    name="study_run",
    exist_ok=True,
)
print("결과 저장 폴더:", train_results.save_dir)

### 3-1. 학습 결과 지표 보기
학습이 끝나면 `save_dir`에 그래프·CSV·가중치가 쌓입니다. `results.csv`를 표로 봅니다.

In [ ]:
import pandas as pd

csv_path = f"{train_results.save_dir}/results.csv"
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)", "train/box_loss"]]

## 4. 검증(Validate)
학습에 쓴 데이터셋의 val 세트로 성능(mAP)을 측정합니다.

In [ ]:
metrics = model.val(device=DEVICE)
print("mAP50-95:", round(metrics.box.map, 4))
print("mAP50   :", round(metrics.box.map50, 4))
print("클래스별 mAP50-95:", [round(x, 3) for x in metrics.box.maps])

## 5. 내보내기(Export)
배포용 포맷으로 변환합니다. `onnx`, `torchscript`, `coreml`(Apple), `tflite` 등 지원.

In [ ]:
path = model.export(format="onnx")
print("내보낸 파일:", path)

## 6. 자유 실험 공간

여기부터는 마음껏 실험해 보세요. 아이디어:
- `conf`, `iou` 값을 바꿔가며 탐지 결과 변화 관찰
- `yolo26s.pt` 등 더 큰 모델로 정확도/속도 비교
- 세그멘테이션(`yolo26n-seg.pt`), 포즈(`yolo26n-pose.pt`) 등 다른 task 시도
- 본인 이미지 폴더로 배치 추론
- `model.track(...)` 로 비디오 객체 추적

In [ ]:
# 예시: conf 값을 낮춰 더 많이 탐지해보기
r2 = model.predict("./image.png", device=DEVICE, conf=0.1)
print("탐지 수:", len(r2[0].boxes))
im_bgr = r2[0].plot()
Image.fromarray(im_bgr[..., ::-1])   # BGR -> RGB 로 뒤집어 표시